# Correspondência de endereços

Gera candidatos, calcula similaridades, decide o resultado por consulta, permite auditar casos incertos e exporta a base final.


In [17]:
# Objetivo: Abre as bases silver no DuckDB, prepara a pasta gold e cria as views usadas nas consultas.

from pathlib import Path
import duckdb

# Define as pastas das bases intermediarias e dos resultados finais.
SILVER = Path("../data/silver")
GOLD = Path("../data/gold")
GOLD.mkdir(parents=True, exist_ok=True)

# Abre uma conexao DuckDB em memoria para as consultas do notebook.
con = duckdb.connect()

# Cria uma view CNEFE e deriva o municipio usado para bloquear candidatos.
# Cria uma view de busca com identificador unico por consulta.
con.sql(f"""
    CREATE VIEW cnefe AS
    SELECT *, LEFT(COD_MUNICIPIO, 6) AS municipio_busca
    FROM read_parquet('{SILVER / 'cnefe_ba.parquet'}')
""")

con.sql(f"""
    CREATE VIEW busca AS
    SELECT ROW_NUMBER() OVER () AS id_consulta, *
    FROM read_parquet('{SILVER / 'busca.parquet'}')
""")

print("Views criadas. Busca:", con.sql("SELECT COUNT(*) FROM busca").fetchone()[0],
      "| CNEFE:", con.sql("SELECT COUNT(*) FROM cnefe").fetchone()[0])

Views criadas. Busca: 107 | CNEFE: 3889595


In [18]:
# Objetivo: Gera o conjunto de candidatos por municipio usando CEP exato, prefixo de CEP ou inicio do logradouro.

# ---------------------------------------------------------------------------
# Pool de candidatos: uniao de 3 blocos de chave (deduplicado por par)
# ---------------------------------------------------------------------------
# Une os blocos de candidatos e elimina pares repetidos.
con.sql("""
CREATE OR REPLACE TEMP TABLE candidatos AS
-- Bloco 1: CEP exato
SELECT b.id_consulta, c.COD_UNICO_ENDERECO, 'cep' AS bloco
FROM busca b JOIN cnefe c
  ON b.consulta_municipio = c.municipio_busca
 AND b.cep_limpo = c.cep_limpo
WHERE b.cep_limpo IS NOT NULL

UNION

-- Bloco 2: mesma area postal (cep5) - tolera 1 digito errado ou CEP generico
SELECT b.id_consulta, c.COD_UNICO_ENDERECO, 'cep5' AS bloco
FROM busca b JOIN cnefe c
  ON b.consulta_municipio = c.municipio_busca
 AND b.cep5 = c.cep5
WHERE b.cep_limpo IS NOT NULL

UNION

-- Bloco 3: mesmo inicio de logradouro (sem tipo) - CEP errado/ausente no CNEFE
SELECT b.id_consulta, c.COD_UNICO_ENDERECO, 'core' AS bloco
FROM busca b JOIN cnefe c
  ON b.consulta_municipio = c.municipio_busca
 AND b.k_core = c.k_core
WHERE b.k_core IS NOT NULL AND b.k_core <> ''
""")

print("Pares candidatos:", con.sql("SELECT COUNT(*) FROM candidatos").fetchone()[0])
display(con.sql("SELECT bloco, COUNT(*) AS n FROM candidatos GROUP BY bloco ORDER BY n DESC").df())

Pares candidatos: 3180536


,bloco,n
0,cep5,1632325
1,cep,1429795
2,core,118416


In [19]:
# Objetivo: Calcula similaridades e notas por campo, ajustando os pesos aos dados disponiveis em cada par.

# ---------------------------------------------------------------------------
# Scoring composto por par candidato (renormalizado por campo disponivel)
# - peso do numero so sai do denominador quando AMBOS os lados estao sem numero
# - numero proximo (off-by-one) penaliza forte: divisor 10
# ---------------------------------------------------------------------------
# Calcula similaridades e notas por atributo para cada par candidato.
# Soma as notas com pesos renormalizados conforme os campos disponiveis.
con.sql("""
CREATE OR REPLACE TEMP TABLE scored AS
SELECT
    b.id_consulta,
    c.COD_UNICO_ENDERECO,
    c.COD_SETOR,
    c.LOGRADOURO            AS match_logradouro,
    c.NUMERO                AS match_numero,
    c.LOCALIDADE            AS match_localidade,
    c.CEP                   AS match_cep,

    jaro_winkler_similarity(b.logr_norm, c.logr_norm) AS sim_logr,
    jaro_winkler_similarity(b.logr_core, c.logr_core) AS sim_core,

    CASE WHEN b.bairro_norm IS NULL OR c.localidade_norm IS NULL THEN NULL
         ELSE jaro_winkler_similarity(b.bairro_norm, c.localidade_norm)
    END AS sim_bairro,

    CASE WHEN b.comp_norm IS NOT NULL AND c.comp_norm IS NOT NULL
         THEN jaro_winkler_similarity(b.comp_norm, c.comp_norm)
    END AS sim_comp,

    (b.cep_limpo IS NOT NULL) AS cep_avail,
    CASE WHEN b.cep_limpo IS NOT NULL AND b.cep_limpo = c.cep_limpo THEN 1.0
         WHEN b.cep5 IS NOT NULL AND b.cep5 = c.cep5 THEN 0.6
         ELSE 0.0
    END AS cep_score,

    NOT (b.consulta_numero IS NULL AND c.NUMERO IS NULL) AS num_avail,

    CASE
        WHEN b.consulta_numero IS NOT NULL AND c.NUMERO IS NOT NULL
             AND TRY_CAST(b.consulta_numero AS INTEGER) = TRY_CAST(c.NUMERO AS INTEGER)
            THEN 1.0
        WHEN b.consulta_numero IS NULL AND c.NUMERO IS NULL THEN 0.6
        WHEN b.consulta_numero IS NULL OR c.NUMERO IS NULL THEN 0.4
        ELSE GREATEST(0.0, 1.0 - ABS(TRY_CAST(b.consulta_numero AS INTEGER)
                                   - TRY_CAST(c.NUMERO AS INTEGER)) / 10.0)
    END AS num_score

FROM candidatos k
JOIN busca  b USING (id_consulta)
JOIN cnefe  c USING (COD_UNICO_ENDERECO)
WHERE jaro_winkler_similarity(b.logr_norm, c.logr_norm) >= 0.50
""")

con.sql("""
CREATE OR REPLACE TEMP TABLE scored2 AS
SELECT *,
    ROUND(
        ( 0.42 * sim_logr
        + 0.13 * sim_core
        + CASE WHEN sim_bairro IS NOT NULL THEN 0.15 * sim_bairro ELSE 0 END
        + CASE WHEN cep_avail             THEN 0.05 * cep_score  ELSE 0 END
        + CASE WHEN num_avail             THEN 0.20 * num_score  ELSE 0 END
        + CASE WHEN sim_comp   IS NOT NULL THEN 0.05 * sim_comp  ELSE 0 END
        )
        /
        ( 0.42 + 0.13
        + CASE WHEN sim_bairro IS NOT NULL THEN 0.15 ELSE 0 END
        + CASE WHEN cep_avail             THEN 0.05 ELSE 0 END
        + CASE WHEN num_avail             THEN 0.20 ELSE 0 END
        + CASE WHEN sim_comp   IS NOT NULL THEN 0.05 ELSE 0 END
        )
    , 4) AS score
FROM scored
""")

con.sql("CREATE OR REPLACE TEMP TABLE scored AS SELECT * FROM scored2")

print("Pares scored:", con.sql("SELECT COUNT(*) FROM scored").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pares scored: 3012972


In [20]:
# Objetivo: Remove duplicatas, compara os dois melhores candidatos por consulta e atribui o status do matching.

# ---------------------------------------------------------------------------
# Dedup por endereco + Top-2 por consulta + regra de decisao
# ---------------------------------------------------------------------------
# Deduplica endereços equivalentes e mantem ate dois candidatos por consulta.
# Consolida os melhores campos e notas de cada consulta.
con.sql("""
CREATE OR REPLACE TEMP TABLE scored_dedup AS
SELECT * EXCLUDE (dup_rn)
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY id_consulta,
                         UPPER(TRIM(match_logradouro)),
                         COALESCE(TRY_CAST(match_numero AS BIGINT), -1)
            ORDER BY score DESC, sim_logr DESC
        ) AS dup_rn
    FROM scored
)
WHERE dup_rn = 1
""")

con.sql("""
CREATE OR REPLACE TEMP TABLE melhores AS
SELECT *
FROM scored_dedup
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id_consulta
    ORDER BY score DESC, num_score DESC, sim_logr DESC
) <= 2
""")

con.sql("""
CREATE OR REPLACE TEMP TABLE decisao AS
SELECT
    id_consulta,
    MAX(CASE WHEN rn = 1 THEN COD_SETOR END)        AS setor_censitario_encontrado,
    MAX(CASE WHEN rn = 1 THEN score END)            AS score_top1,
    MAX(CASE WHEN rn = 1 THEN sim_logr END)         AS sim_top1,
    MAX(CASE WHEN rn = 1 THEN sim_core END)         AS sim_core_top1,
    MAX(CASE WHEN rn = 1 THEN num_score END)        AS num_top1,
    MAX(CASE WHEN rn = 2 THEN score END)            AS score_top2,
    MAX(CASE WHEN rn = 1 THEN match_logradouro END) AS match_logradouro,
    MAX(CASE WHEN rn = 1 THEN match_numero END)     AS match_numero,
    MAX(CASE WHEN rn = 1 THEN match_localidade END) AS match_localidade,
    MAX(CASE WHEN rn = 1 THEN match_cep END)        AS match_cep
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY id_consulta
                                 ORDER BY score DESC, num_score DESC, sim_logr DESC) AS rn
    FROM melhores
) t
GROUP BY id_consulta
""")

# Aplica as regras de decisao e cria o resultado completo da busca.
con.sql(f"""
CREATE OR REPLACE TEMP TABLE resultado AS
SELECT
    b.id_consulta,
    b.consulta_logradouro,
    b.consulta_numero,
    b.consulta_bairro,
    b.consulta_cep,
    b.consulta_complemento,
    b.consulta_municipio,
    d.setor_censitario_encontrado,
    ROUND(d.score_top1, 3) AS score_match,
    d.match_logradouro,
    d.match_numero,
    d.match_localidade,
    d.match_cep,
    CASE
        WHEN d.id_consulta IS NULL THEN 'nao_encontrado'

        -- TRAVA 1: consulta tem numero, match nao tem -> nunca 'encontrado'
        WHEN b.consulta_numero IS NOT NULL AND d.match_numero IS NULL
            THEN 'ambiguo'

        -- TRAVA 2: consulta tem numero -> exige numero EXATO no match
        WHEN b.consulta_numero IS NOT NULL AND d.num_top1 < 1.0
            THEN 'ambiguo'

        -- numero exato + rua quase igual no CORE (sem tipo, nao engana com prefixo)
        WHEN d.num_top1 = 1.0
         AND d.sim_core_top1 >= 0.90
         AND d.sim_top1 >= 0.90
            THEN 'encontrado'

        -- numero exato + score alto com folga sobre o rival
        WHEN d.num_top1 = 1.0 AND d.score_top1 >= 0.85
             AND (d.score_top2 IS NULL OR d.score_top1 - d.score_top2 >= 0.03)
            THEN 'encontrado'

        -- consulta sem numero: core muito alto + folga
        WHEN b.consulta_numero IS NULL
         AND d.sim_core_top1 >= 0.95
         AND (d.score_top2 IS NULL OR d.score_top1 - d.score_top2 >= 0.02)
            THEN 'encontrado'

        WHEN d.score_top1 >= 0.65 THEN 'ambiguo'
        ELSE 'nao_encontrado'
    END AS status_match
FROM busca b
LEFT JOIN decisao d USING (id_consulta)
ORDER BY b.id_consulta
""")

display(con.sql("SELECT status_match, COUNT(*) AS n FROM resultado GROUP BY status_match ORDER BY n DESC").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,status_match,n
0,ambiguo,79
1,encontrado,28


In [21]:
# Objetivo: Lista resultados ambiguos ou sem correspondencia para facilitar a revisão manual.

# ---------------------------------------------------------------------------
# Auditoria: casos ambiguos e não encontrados (melhor candidato para revisão)
# ---------------------------------------------------------------------------
# Exibe os casos que exigem revisão por ambiguidade ou falta de correspondencia.
display(con.sql("""
    SELECT id_consulta, consulta_logradouro, consulta_numero, consulta_bairro,
           consulta_cep, status_match, score_match,
           match_logradouro, match_numero, match_localidade
    FROM resultado
    WHERE status_match IN ('ambiguo', 'nao_encontrado')
    ORDER BY score_match DESC NULLS LAST
""").df())

,id_consulta,consulta_logradouro,consulta_numero,consulta_bairro,consulta_cep,status_match,score_match,match_logradouro,match_numero,match_localidade
0,89,RUA ILHEUS,17,MORADA DO PLANALTO,45190000,ambiguo,0.979,RUA ILHEUS,18,MORADA DO PLANALTO
1,31,RUA ALZIRO ROCHA,237,None,48120000,ambiguo,0.975,RUA ALZIRO ROCHA,236,INOCOOP
2,15,RUA NOVA,23,None,45745000,ambiguo,0.975,RUA NOVA,22,BELA VISTA
3,43,RUA AZEREDO COUTINHO,49,None,40370330,ambiguo,0.975,RUA AZEREDO COUTINHO,48,LIBERDADE
4,14,FAZENDA BOA VISTA,None,SAO JOAO DO SUL,45840000,ambiguo,0.965,FAZENDA BELA VISTA,None,SAO JOAO DO SUL
...,...,...,...,...,...,...,...,...,...,...
74,25,DISTRITO ENG PONTES,None,BELA VITA,45490000,ambiguo,0.739,ESTRADA ENGENHEIRO PONTES,None,DEUS DARA
75,47,PONTA QUEIMADAS,None,RURAL,48700000,ambiguo,0.737,TRAVESSA QUEIMADAS,None,CIDADE NOVA
76,66,DISTRITO TAGUA,228,TAGUA,47900000,ambiguo,0.732,RUA DA MATRIZ,228,TAGUA
77,67,SITIO PECUARIA,315,None,48480000,ambiguo,0.729,ESTRADA UMBUZEIRO PARA UMBAUBA,315,UMBUZEIRO


In [22]:
# Objetivo: Seleciona as sete colunas de entrega, exporta CSV e Parquet e resume os resultados para auditoria.

# ---------------------------------------------------------------------------
# Exportação Gold (CSV + Parquet)
# Apenas as 7 colunas solicitadas
# ---------------------------------------------------------------------------

# Mantem somente as colunas previstas para a entrega final.
con.sql("""
CREATE OR REPLACE TEMP VIEW resultado_gold AS
SELECT
    consulta_logradouro,
    consulta_numero,
    consulta_bairro,
    consulta_cep,
    consulta_complemento,
    consulta_municipio,
    setor_censitario_encontrado
FROM resultado
ORDER BY id_consulta
""")

# Grava a entrega em CSV e Parquet para consumo posterior.
# Exportar CSV
con.sql(f"""
    COPY resultado_gold
    TO '{GOLD / 'resultado_final.csv'}'
    (HEADER, DELIMITER ';')
""")

# Exportar Parquet
con.sql(f"""
    COPY resultado_gold
    TO '{GOLD / 'resultado_final.parquet'}'
    (FORMAT PARQUET)
""")

# Confere o caminho, as dimensões, uma amostra e a distribuição dos status.
print("Resultado salvo em:", GOLD)

print("\nDimensões da Gold:")
print(con.sql("SELECT COUNT(*) AS total FROM resultado_gold").df())

print("\nPrévia da Gold:")
display(con.sql("SELECT * FROM resultado_gold LIMIT 10").df())

print("\nDistribuição dos matches (auditoria interna):")
display(con.sql("""
    SELECT status_match, COUNT(*) AS total
    FROM resultado
    GROUP BY status_match
    ORDER BY total DESC
""").df())

Resultado salvo em: ..\data\gold

Dimensões da Gold:
   total
0    107

Prévia da Gold:


,consulta_logradouro,consulta_numero,consulta_bairro,consulta_cep,consulta_complemento,consulta_municipio,setor_censitario_encontrado
0,RODOVIA BR 101,None,BR101,48030260,None,290070,290070205000112
1,RUA RIO BAIANO,None,CENTRO,46830000,None,290130,290130405000002
2,FAZENDA BICO,None,None,46130000,None,290200,290200505000015
3,DISTRITO 2 DE JULHO,22,POCO CENTRAL,45675000,None,290240,290240115000001
4,RUA SERGIO BOMFIM,332,BARREIRINHAS,47800020,None,290320,290320105000030
5,EVILASIO FIRMATO,28,SOCRATES REZENDE,45860000,None,290630,290630305000039
6,FAZENDA LAGOA GRANDE,None,None,46380000,None,290660,290660005000007
7,CONJUNTO URBES NOVA,2,NOVA PASTORA,48390000,None,290700,290700405000002
8,CONDOMINIO RESI DOUTOR ANTONIO CARLOS GOES,1,None,48110000,None,290750,290750905000089
9,RUA CASSIMIRO DE ABREU,89,None,45822000,None,291072,291072705000137



Distribuição dos matches (auditoria interna):


,status_match,total
0,ambiguo,79
1,encontrado,28
